# NOUL as a plain neural-network problem

`decoder_jepa.ipynb` gave the same answer to every NOUL question, whatever the targets. This notebook asks the simplest version of the question: **given the input and the designed output, can a deep MLP learn the mapping?**

- **Input:** each NOUL question's text, built exactly as in `decoder_jepa` (state, question, candidates), encoded **once** by frozen ModernBERT-base into a 768-d vector (mean over all tokens, and the `[CLS]` vector).
- **Output:** the designed targets, ModernBERT's encoding of `yes` × 64 (true) and `no` × 64 (false), mean over the word tokens. The MLP outputs a vector ẑ and the answer is `softmax(cos(ẑ, t_yes), cos(ẑ, t_no)) / τ`, trained with KL to the gold.
- **Model:** a deep residual MLP. Several depths and learning rates are tried; they train in seconds because the features are computed once.

**How to read it:**
- **Fits training but not validation:** the MLP can learn, but frozen ModernBERT features don't carry the answer in a way that generalises (fine-tuning or another encoder is needed).
- **Cannot fit training:** something upstream is wrong (inputs or targets).
- **Beats uniform on validation:** the earlier runs were held back by fine-tuning or optimisation, not by the design.

A **linear classifier** (2 logits) on the same features is trained alongside, as a reference.

## Settings

In [ ]:
import os


def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        kind = type(default[0]) if default else str
        return [kind(v.strip()) for v in value.split(",") if v.strip()]
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    return type(default)(value)


CONFIG = {
    "seed": setting("seed", 42),
    "model_name": setting("model_name", "answerdotai/ModernBERT-base"),
    "data_path": setting("data_path", "data/signpost_d_decisions/train.jsonl"),
    "data_url": setting("data_url", "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                                    "decider-jepa/data/signpost_d_decisions/train.jsonl"),
    "max_sequence_length": setting("max_sequence_length", 1024),
    "split_seed": setting("split_seed", 42),
    "train_ratio": setting("train_ratio", 0.80),
    "validation_ratio": setting("validation_ratio", 0.10),

    # the designed targets
    "target_words": setting("target_words", ["yes", "no"]),
    "target_repeats": setting("target_repeats", 64),
    "temperature": setting("temperature", 0.1),

    # the MLP search
    "poolings": setting("poolings", ["mean", "cls"]),
    "depths": setting("depths", [2, 4, 8]),           # residual blocks
    "width": setting("width", 1024),
    "learning_rates": setting("learning_rates", [1e-3, 3e-4]),
    "dropout": setting("dropout", 0.1),
    "weight_decay": setting("weight_decay", 0.01),
    "epochs": setting("epochs", 300),
    "batch_size": setting("batch_size", 256),
    "memorise_n": setting("memorise_n", 200),         # the memorisation check: this many training questions

    "encode_batch": setting("encode_batch", 16),
    "td_dataset": setting("td_dataset", "LocalLLaMA/typed-decisions"),
    "td_file": setting("td_file", "all/test-00000-of-00001.parquet"),
    "out_dir": setting("out_dir", "mlp_check"),
}

for k, v in CONFIG.items():
    print(f"{k:22s}: {v}")

## Data: NOUL questions and the group-aware split (as in `decoder_jepa`)

In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PATH = Path(CONFIG["data_path"])
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["data_url"], DATA_PATH)


def get_candidates(question):
    """decoder_jepa's rule: a NOUL question without criteria gets the literal true / false."""
    criteria = question.get("criteria")
    if criteria is None:
        return [{"key": "true", "description": "true"}, {"key": "false", "description": "false"}]
    return [{"key": str(k), "description": str(v)} for k, v in criteria.items()]


def online_text(state, instructions, candidates):
    """decoder_jepa's online input, unchanged."""
    return ("State:\n" + str(state) + "\n\nQuestion:\n" + str(instructions) + "\n\nCandidates:\n"
            + "\n".join(f'{c["key"]}: {c["description"]}' for c in candidates))


def noul_samples(records, group_key):
    out = []
    for record in records:
        for qid, q in record["questions"].items():
            if str(q.get("type", "")).lower() != "noul":
                continue
            candidates = get_candidates(q)
            gold = {str(k): float(v) for k, v in record["gold"][qid]["probabilities"].items()}
            assert {c["key"] for c in candidates} == {"true", "false"} == set(gold)
            out.append({"group": record[group_key], "question_id": qid,
                        "text": online_text(record["state"], q.get("instructions"), candidates),
                        "gold": [gold["true"], gold["false"]]})
    return out


records = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]
samples = noul_samples(records, "id")

groups = pd.Series([s["group"] for s in samples]).drop_duplicates().to_numpy()
shuffled = np.random.default_rng(CONFIG["split_seed"]).permutation(groups)
n_train = int(len(groups) * CONFIG["train_ratio"])
n_val = int(len(groups) * CONFIG["validation_ratio"])
split_of = {g: "train" for g in shuffled[:n_train]}
split_of.update({g: "validation" for g in shuffled[n_train:n_train + n_val]})
split_of.update({g: "test" for g in shuffled[n_train + n_val:]})

splits = {name: [s for s in samples if split_of[s["group"]] == name] for name in ("train", "validation", "test")}
print({k: len(v) for k, v in splits.items()})

## Typed Decisions test: the 600 NOUL questions

In [ ]:
from huggingface_hub import hf_hub_download

td_frame = pd.read_parquet(hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset"))


def parsed(value):
    return json.loads(value) if isinstance(value, str) else value


td_records = [{"id": r["id"], "state": parsed(r["state"]), "questions": parsed(r["questions"]),
               "gold": parsed(r["gold"])} for r in td_frame.to_dict("records")]
splits["td_test"] = noul_samples(td_records, "id")
print("Typed Decisions NOUL questions:", len(splits["td_test"]))

## Frozen ModernBERT: encode every input once, and the two targets

In [ ]:
import math

import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
encoder = AutoModel.from_pretrained(CONFIG["model_name"]).to(device).eval()


@torch.no_grad()
def encode_texts(texts):
    """-> {"mean": [n, 768], "cls": [n, 768]}, batches of similar length."""
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    mean = torch.zeros(len(texts), encoder.config.hidden_size)
    cls = torch.zeros(len(texts), encoder.config.hidden_size)
    for s in range(0, len(order), CONFIG["encode_batch"]):
        idx = order[s:s + CONFIG["encode_batch"]]
        batch = tokenizer([texts[i] for i in idx], padding=True, truncation=True,
                          max_length=CONFIG["max_sequence_length"], return_tensors="pt").to(device)
        states = encoder(**batch).last_hidden_state.float()
        mask = batch["attention_mask"].unsqueeze(-1).float()
        mean[idx] = ((states * mask).sum(1) / mask.sum(1)).cpu()
        cls[idx] = states[:, 0].cpu()
    return {"mean": mean, "cls": cls}


features = {name: encode_texts([s["text"] for s in rows]) for name, rows in splits.items()}
golds = {name: torch.tensor([s["gold"] for s in rows]) for name, rows in splits.items()}


@torch.no_grad()
def encode_answer(word):
    text = " ".join([word] * CONFIG["target_repeats"])
    batch = tokenizer(text, return_tensors="pt", return_special_tokens_mask=True)
    special = batch.pop("special_tokens_mask")[0].bool()
    states = encoder(**batch.to(device)).last_hidden_state[0].float().cpu()
    return states[~special].mean(0)


yes_word, no_word = CONFIG["target_words"]
TARGETS = torch.stack([encode_answer(yes_word), encode_answer(no_word)]).to(device)   # [true, false]
c = F.cosine_similarity(TARGETS[0], TARGETS[1], dim=0).item()
tau = CONFIG["temperature"]
print(f"targets: {yes_word} / {no_word} x {CONFIG['target_repeats']} | cos {c:.4f} | "
      f"highest gold probability {1 / (1 + math.exp(-(1 - c) / tau)):.4f}")
print({k: tuple(v["mean"].shape) for k, v in features.items()})

## Quick look: do the features differ between questions at all?

If every input's feature vector were nearly the same, no head could learn. The average cosine between different questions' features, and how much of each feature varies across questions.

In [ ]:
for pooling in CONFIG["poolings"]:
    x = features["train"][pooling]
    xn = F.normalize(x, dim=1)
    sample = xn[torch.randperm(len(xn))[:500]]
    sim = sample @ sample.T
    off = sim[~torch.eye(len(sample), dtype=torch.bool)]
    print(f"{pooling:5s}: mean cosine between different questions {off.mean():.4f} "
          f"(min {off.min():.4f}) | per-dimension std / mean |x| {x.std(0).mean() / x.abs().mean():.4f}")

## The models

- **MLP:** input → width, then `depth` residual blocks (LayerNorm → Linear → GELU → Dropout → Linear), then LayerNorm → Linear to the target size (768). The answer: softmax(cos(ẑ, targets) / τ).
- **Linear classifier (reference):** one Linear layer to 2 logits, softmax.

Inputs are standardised with the training split's mean and std. Both are trained with KL to the gold.

In [ ]:
import torch.nn as nn


class Block(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(width, width))

    def forward(self, x):
        return x + self.net(x)


class TargetMLP(nn.Module):
    def __init__(self, depth, width, dropout, out_dim):
        super().__init__()
        self.inp = nn.Linear(features["train"]["mean"].shape[1], width)
        self.blocks = nn.Sequential(*[Block(width, dropout) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, out_dim))

    def forward(self, x):
        z = self.out(self.blocks(self.inp(x)))
        return F.cosine_similarity(z.unsqueeze(1), TARGETS.unsqueeze(0), dim=-1) / CONFIG["temperature"]


class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(features["train"]["mean"].shape[1], 2)

    def forward(self, x):
        return self.lin(x)


def kl(logits, gold):
    logp = F.log_softmax(logits, dim=1)
    return (gold * (gold.clamp_min(1e-12).log() - logp)).sum(1)


UNIFORM = torch.tensor([0.5, 0.5])


def measures(logits, gold):
    p = F.softmax(logits, dim=1)
    return {
        "kl": kl(logits, gold).mean().item(),
        "uniform_kl": kl(torch.log(UNIFORM.to(gold.device)).expand_as(gold), gold).mean().item(),
        "top_choice": (p.argmax(1) == gold.argmax(1)).float().mean().item(),
        "brier": ((p - gold) ** 2).sum(1).mean().item(),
        "p_true_std": p[:, 0].std().item(),
        "p_true_min": p[:, 0].min().item(),
        "p_true_max": p[:, 0].max().item(),
    }

## Training one setting

In [ ]:
import copy
import time


def standardise(pooling):
    mu = features["train"][pooling].mean(0)
    sd = features["train"][pooling].std(0).clamp_min(1e-6)
    return {name: ((f[pooling] - mu) / sd).to(device) for name, f in features.items()}


def train(model, x_train, g_train, x_val, g_val, lr, epochs):
    torch.manual_seed(CONFIG["seed"])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best = (float("inf"), None, 0)
    g_train = g_train.to(device)
    g_val = g_val.to(device) if g_val is not None else None
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(x_train), device=device)
        for s in range(0, len(perm), CONFIG["batch_size"]):
            idx = perm[s:s + CONFIG["batch_size"]]
            loss = kl(model(x_train[idx]), g_train[idx]).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sched.step()
        if x_val is not None:
            model.eval()
            with torch.no_grad():
                v = kl(model(x_val), g_val).mean().item()
            if v < best[0]:
                best = (v, copy.deepcopy(model.state_dict()), epoch)
    if best[1] is not None:
        model.load_state_dict(best[1])
    return model, best[2]


@torch.no_grad()
def evaluate(model, x, g):
    model.eval()
    return measures(model(x), g.to(device))

## 1. Memorisation check

Train on only `memorise_n` training questions, no validation, no dropout, and see whether the training KL falls far below uniform. If it can't, something upstream is wrong.

In [ ]:
memorise_rows = []

for pooling in CONFIG["poolings"]:
    X = standardise(pooling)
    n = CONFIG["memorise_n"]
    x, g = X["train"][:n], golds["train"][:n]
    for depth in CONFIG["depths"]:
        model = TargetMLP(depth, CONFIG["width"], 0.0, TARGETS.shape[1]).to(device)
        model, _ = train(model, x, g, None, None, CONFIG["learning_rates"][0], CONFIG["epochs"])
        memorise_rows.append({"pooling": pooling, "model": f"MLP depth {depth}", **evaluate(model, x, g)})
    model, _ = train(LinearClassifier().to(device), x, g, None, None, CONFIG["learning_rates"][0], CONFIG["epochs"])
    memorise_rows.append({"pooling": pooling, "model": "linear classifier", **evaluate(model, x, g)})

memorise = pd.DataFrame(memorise_rows)
display(memorise.round(4))

## 2. The full search: train on the training split, select on validation

Every pooling × depth × learning rate. The best epoch by validation KL is kept. Measures on the validation split, our test split, and the 600 Typed Decisions NOUL questions.

In [ ]:
rows, models = [], {}
t0 = time.perf_counter()

for pooling in CONFIG["poolings"]:
    X = standardise(pooling)
    for lr in CONFIG["learning_rates"]:
        candidates = [(f"MLP depth {d}", lambda d=d: TargetMLP(d, CONFIG["width"], CONFIG["dropout"],
                                                              TARGETS.shape[1])) for d in CONFIG["depths"]]
        candidates.append(("linear classifier", LinearClassifier))
        for name, make in candidates:
            model, best_epoch = train(make().to(device), X["train"], golds["train"], X["validation"],
                                      golds["validation"], lr, CONFIG["epochs"])
            key = (pooling, name, lr)
            models[key] = model
            row = {"pooling": pooling, "model": name, "lr": lr, "best_epoch": best_epoch}
            for split in ("train", "validation", "test", "td_test"):
                m = evaluate(model, X[split], golds[split])
                row.update({f"{split}_{k}": v for k, v in m.items()
                            if k in ("kl", "uniform_kl", "top_choice", "p_true_std")})
            rows.append(row)

results = pd.DataFrame(rows).sort_values("validation_kl").reset_index(drop=True)
print(f"{len(results)} settings in {time.perf_counter() - t0:.0f} s")

pd.set_option("display.width", 250)
display(results[["pooling", "model", "lr", "best_epoch",
                 "train_kl", "train_uniform_kl", "train_top_choice",
                 "validation_kl", "validation_uniform_kl", "validation_top_choice", "validation_p_true_std",
                 "td_test_top_choice", "td_test_kl", "td_test_p_true_std"]].round(4))

## Verdict

In [ ]:
best = results.iloc[0]
val_gain = best["validation_uniform_kl"] - best["validation_kl"]
train_gain = memorise[memorise["model"].str.startswith("MLP")].eval("uniform_kl - kl").max()

print("Best setting:", best["pooling"], best["model"], "lr", best["lr"])
print(f"  memorisation: best training-KL gain over uniform {train_gain:.4f}")
print(f"  validation  : KL {best['validation_kl']:.4f} vs uniform {best['validation_uniform_kl']:.4f} "
      f"(gain {val_gain:.4f}), top-choice {best['validation_top_choice']:.4f}, "
      f"P(true) std {best['validation_p_true_std']:.4f}")
print(f"  Typed Decisions NOUL: top-choice {best['td_test_top_choice']:.4f}, KL {best['td_test_kl']:.4f} "
      f"(Signpost-D-4B: 0.7833 / 0.0952)")

if train_gain < 0.05:
    print("\nVERDICT: the MLP cannot even fit the training questions -> look upstream (inputs / targets).")
elif val_gain < 0.005:
    print("\nVERDICT: the MLP fits training but does not generalise -> frozen ModernBERT features don't carry the answer.")
else:
    print("\nVERDICT: the MLP generalises beyond uniform -> the earlier runs were held back by fine-tuning / optimisation.")

os.makedirs(CONFIG["out_dir"], exist_ok=True)
results.to_csv(os.path.join(CONFIG["out_dir"], "mlp_search.csv"), index=False)
memorise.to_csv(os.path.join(CONFIG["out_dir"], "memorise.csv"), index=False)
print("\nSaved:", CONFIG["out_dir"])